# FEMTO/PRONOSTIA Bearing Analysis — Approximate OEM Priors

The FEMTO/PRONOSTIA dataset from IEEE PHM 2012 Challenge contains 15 ball bearing run-to-failure experiments across three operating conditions (varying RPM and radial load). Unlike IMS where exact bearing specs are known, the FEMTO bearing model is undocumented in the public literature. This notebook quantifies how well the adaptive PID framework performs when the OEM prior is approximate — the key generalization question.

## 1. Setup

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path

# Colorblind-friendly palette
CB = ["#0072B2", "#E69F00", "#009E73", "#CC79A7", "#56B4E9", "#D55E00"]
sns.set_style("whitegrid")
plt.rcParams.update({"figure.figsize": (12, 6), "font.size": 12})

FIGURES_DIR = Path("../reports/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
ANALYSIS_DIR = Path("../analysis")
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

from datasets.femto.config import FEMTO_CONFIG
from datasets.femto.loader import FEMTOLoader, BEARING_MAP
from framework.benchmark_runner import run_single_trajectory, run_dataset
from framework.dataset_loader import OEMPrior
from core.adaptive_drift import adaptive_drift_pid, adaptive_drift_with_regime, PIDParams

## 2. Load FEMTO Data

In [ ]:
loader = FEMTOLoader()
trajectories = loader.load_trajectories()

print(f"Loaded {len(trajectories)} trajectories")

# Summarise by condition and split
summary_rows = []
for traj in trajectories:
    meta = traj.metadata
    summary_rows.append({
        "bearing_id": meta["bearing_id"],
        "condition": meta["condition"],
        "split": meta["split"],
        "n_steps": len(traj.features),
        "has_rul": traj.true_rul is not None,
        "prior_source": traj.oem_prior.source if traj.oem_prior else "none",
        "prior_life_h": round(traj.oem_prior.expected_life, 1) if traj.oem_prior else None,
    })

summary_df = pd.DataFrame(summary_rows)
print("\nDataset inventory:")
print(summary_df.to_string(index=False))

## 3. Aggregate Results — All Trajectories

In [ ]:
all_results = []
for traj in trajectories:
    result = run_single_trajectory(traj)
    all_results.append(result)

results_df = pd.concat(all_results, ignore_index=True)

# Aggregate by model
agg = (
    results_df
    .groupby("model")[["rmse", "mae", "nasa_score", "detection_lead_time", "false_alarm_rate"]]
    .agg(["mean", "median", "std"])
    .round(2)
)

print("Aggregate benchmark results — FEMTO (all trajectories, training + test):")
print(agg.to_string())

## 4. Per-Condition Comparison

In [ ]:
# Attach condition metadata to results
condition_map = {
    f"femto_{bid}": info["condition"]
    for bid, info in BEARING_MAP.items()
}
results_df["condition"] = results_df["unit_id"].map(condition_map)

CONDITIONS = {
    1: "Cond 1 (1800 rpm / 4000 N)",
    2: "Cond 2 (1650 rpm / 4200 N)",
    3: "Cond 3 (1500 rpm / 5000 N)",
}

pid_models = ["pid_adaptive", "pid_regime"]
pid_results = results_df[results_df["model"].isin(pid_models)].copy()

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=False)

for ax, (cond_id, cond_label) in zip(axes, CONDITIONS.items()):
    subset = pid_results[pid_results["condition"] == cond_id]
    if subset.empty:
        ax.set_title(cond_label)
        ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
        continue

    model_order = ["pid_adaptive", "pid_regime"]
    rmse_vals = [
        subset[subset["model"] == m]["rmse"].mean()
        for m in model_order
    ]
    labels = ["PID Adaptive", "PID + Regime"]
    bars = ax.bar(labels, rmse_vals, color=[CB[0], CB[1]], edgecolor="white", linewidth=0.8)
    for bar, val in zip(bars, rmse_vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                f"{val:.1f}", ha="center", va="bottom", fontsize=10)
    ax.set_ylabel("Mean RMSE (hours)" if cond_id == 1 else "")
    ax.set_title(cond_label, fontsize=11)
    ax.set_ylim(0, max(rmse_vals) * 1.3 + 1)

fig.suptitle("PID Model RMSE by Operating Condition — FEMTO", fontsize=13, y=1.01)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "femto_condition_rmse.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Key Question: Approximate Prior vs. Exact Prior

The FEMTO bearing model is undocumented. We use a SKF 6204-2RS as an approximate match based on load capacity. The IMS dataset uses Rexnord ZA-2115 specs (exact). How much does prior quality matter?

In [ ]:
# Print OEM prior details for each condition
print("Approximate OEM priors used (FEMTO):")
print(f"  Comparable model: {FEMTO_CONFIG['approximate_oem_specs']['comparable_model']}")
print(f"  Life exponent (p): {FEMTO_CONFIG['approximate_oem_specs']['life_exponent']}")
print(f"  Estimated C (kN): {FEMTO_CONFIG['approximate_oem_specs']['estimated_C_kn']}")
print()

from core.oem_prior import compute_l10_hours
for cond_id, cond_info in FEMTO_CONFIG["conditions"].items():
    C_kn = FEMTO_CONFIG["approximate_oem_specs"]["estimated_C_kn"]
    P_kn = cond_info["radial_load_N"] / 1000.0
    l10h = compute_l10_hours(C_kn, P_kn, cond_info["rpm"], FEMTO_CONFIG["approximate_oem_specs"]["life_exponent"])
    print(f"  Condition {cond_id} ({cond_info['rpm']} rpm, {cond_info['radial_load_N']} N): L10 = {l10h:.1f} h")

print()
print("Comparison — RMSE (mean across RTF trajectories) by prior quality:")
femto_rtf = results_df[
    (results_df["is_run_to_failure"] == True) &
    (results_df["model"] == "pid_adaptive") &
    results_df["rmse"].notna()
]

try:
    from datasets.ims.loader import IMSLoader
    ims_loader = IMSLoader()
    ims_trajs = ims_loader.load_trajectories()
    ims_all = pd.concat([run_single_trajectory(t) for t in ims_trajs], ignore_index=True)
    ims_pid = ims_all[(ims_all["model"] == "pid_adaptive") & ims_all["rmse"].notna()]
    ims_rmse = ims_pid["rmse"].mean()
    print(f"  IMS (exact OEM prior):     mean RMSE = {ims_rmse:.2f} h")
except Exception as e:
    ims_rmse = None
    print(f"  IMS: could not load — {e}")

femto_rmse = femto_rtf["rmse"].mean()
print(f"  FEMTO (approx OEM prior):  mean RMSE = {femto_rmse:.2f} h")

if ims_rmse:
    delta = femto_rmse - ims_rmse
    print(f"\n  Delta (FEMTO - IMS): {delta:+.2f} h")
    print(f"  The adaptive PID degrades {'moderately' if abs(delta) < 10 else 'substantially'} with an approximate prior.")

## 6. Kurtosis Trajectories — Representative Bearings

In [ ]:
# Pick one representative per condition (prefer training/RTF bearings)
rep_ids = {
    1: "femto_Bearing1_1",
    2: "femto_Bearing2_1",
    3: "femto_Bearing3_3",  # only condition-3 bearing
}

traj_map = {t.unit_id: t for t in trajectories}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (cond_id, unit_id) in zip(axes, rep_ids.items()):
    if unit_id not in traj_map:
        ax.text(0.5, 0.5, f"{unit_id}\nnot available", ha="center", va="center", transform=ax.transAxes)
        continue

    traj = traj_map[unit_id]
    df = traj.features
    primary = traj.primary_feature  # "kurtosis"

    # Time axis in minutes (records are every 10 s)
    if "time" in df.columns:
        t = df["time"].values / 60.0
        xlabel = "Time (min)"
    else:
        t = np.arange(len(df)) * (10 / 3600)  # hours
        xlabel = "Time (h)"

    kurtosis = df[primary].values

    ax.plot(t, kurtosis, color=CB[cond_id - 1], linewidth=0.8, alpha=0.85, label="Kurtosis")

    # Mark OEM prior threshold
    if traj.oem_prior is not None:
        ax.axhline(traj.oem_prior.threshold, color="black", linestyle="--", linewidth=1, label="OEM threshold")

    # Mark failure
    if traj.failure_index is not None:
        t_fail = t[traj.failure_index] if traj.failure_index < len(t) else t[-1]
        ax.axvline(t_fail, color=CB[5], linestyle=":", linewidth=1.2, label="Failure")

    cond_info = FEMTO_CONFIG["conditions"][cond_id]
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Kurtosis" if cond_id == 1 else "")
    ax.set_title(
        f"Bearing {cond_id}_1  [{cond_info['rpm']} rpm / {cond_info['radial_load_N']} N]",
        fontsize=10
    )
    ax.legend(fontsize=8, loc="upper left")

fig.suptitle("Kurtosis Degradation Trajectories — One Bearing per Condition (FEMTO)", fontsize=12, y=1.01)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "femto_kurtosis_trajectories.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Run Full Benchmark and Save Results

In [ ]:
output_path = ANALYSIS_DIR / "femto_metrics.csv"
results_df.to_csv(output_path, index=False)
print(f"Saved {len(results_df)} result rows to {output_path}")

# Clean summary table
clean_summary = (
    results_df
    .groupby(["model", "condition"])[["rmse", "mae", "nasa_score"]]
    .mean()
    .round(2)
)
print("\nMean metrics by model and condition:")
print(clean_summary.to_string())

## 8. Summary: Does the Approximate Prior Work?

Key findings from the FEMTO analysis:

**Prior quality impact:**
- The FEMTO bearing model is undocumented. We use SKF 6204-2RS as a surrogate (approximate match based on load capacity) per the FEMTO_CONFIG.
- Despite approximate L10 estimates, the adaptive PID framework tracks degradation because the PID integral term corrects systematic prior error continuously.
- The regime-detection variant (`pid_regime`) may help in conditions where degradation is non-monotone early in life.

**Per-condition behavior:**
- Condition 3 (highest load, lowest RPM) has only one trajectory (Bearing3_3, test set), so results are illustrative only.
- Conditions 1 and 2 have both training (RTF) and test bearings, enabling a more robust evaluation.

**Comparison to IMS (exact prior):**
- See the cross-dataset comparison above.
- If RMSE degrades moderately (<15–20% relative increase), approximate priors are operationally viable.
- If degradation is large, the recommendation is: collect 1–2 run-to-failure bearings and switch to a fleet-derived prior (as in C-MAPSS notebook 03).